In [1]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
ORDERS = "orders"
ORDER_ITEMS = "order_items"
PRODUCTS = "products"
CUSTOMERS = "customers"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/"

In [2]:
spark = (
        SparkSession.builder.appName("test_order_items")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 23:57:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
order_items = spark.read.format("delta").load(CURATED_PATH + ORDER_ITEMS)
order_items.printSchema()

# get product id with most sales
product_id = order_items.groupBy("product_id").count().orderBy("count", ascending=False).first()[0]
print(product_id)
filter_expression = sf.col("product_id") == product_id


26/03/28 23:58:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sk_product: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_percentage: decimal(10,2) (nullable = true)
 |-- line_total: decimal(10,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



26/03/28 23:59:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


207


In [4]:
products = spark.read.format("delta").load(CURATED_PATH + PRODUCTS)

products.filter(filter_expression).orderBy("product_id", sf.col("created_at").asc()).select("product_id", "sk_product", "effective_from", "effective_to", "is_active").show(10)

order_items.filter(filter_expression).orderBy("order_id", sf.col("created_at").asc()).select("order_id", "product_id", "sk_product", "created_at").show(10)

+----------+----------+-------------------+-------------------+---------+
|product_id|sk_product|     effective_from|       effective_to|is_active|
+----------+----------+-------------------+-------------------+---------+
|       207|       413|2025-01-07 00:00:00|2025-03-15 00:00:00|    false|
|       207|       414|2025-03-15 00:00:00|               NULL|     true|
+----------+----------+-------------------+-------------------+---------+



+--------+----------+----------+-------------------+
|order_id|product_id|sk_product|         created_at|
+--------+----------+----------+-------------------+
|     271|       207|       414|2026-03-27 22:17:49|
|     313|       207|       414|2026-03-27 22:17:49|
|     529|       207|       414|2026-03-27 22:17:49|
|     674|       207|       414|2026-03-27 22:17:49|
|     739|       207|       414|2026-03-27 22:17:49|
|     846|       207|       414|2026-03-27 22:17:49|
|     960|       207|       414|2026-03-27 22:17:49|
|    1208|       207|       414|2026-03-27 22:17:58|
|    1267|       207|       414|2026-03-27 22:17:58|
|    1287|       207|       414|2026-03-27 22:17:58|
+--------+----------+----------+-------------------+
only showing top 10 rows



In [5]:
products.orderBy(sf.col("created_at").desc()).select("product_id", "sk_product", "created_at", "effective_from", "effective_to", "is_active").show(10, truncate=False)

+----------+----------+-------------------+-------------------+-------------------+---------+
|product_id|sk_product|created_at         |effective_from     |effective_to       |is_active|
+----------+----------+-------------------+-------------------+-------------------+---------+
|557       |1114      |2026-03-26 00:00:00|2026-03-26 00:00:00|NULL               |true     |
|778       |1556      |2026-03-26 00:00:00|2026-03-26 00:00:00|NULL               |true     |
|829       |1658      |2026-03-26 00:00:00|2026-03-26 00:00:00|NULL               |true     |
|866       |1732      |2026-03-26 00:00:00|2026-03-26 00:00:00|NULL               |true     |
|218       |436       |2026-03-25 00:00:00|2026-03-25 00:00:00|NULL               |true     |
|962       |1924      |2026-03-25 00:00:00|2026-03-25 00:00:00|NULL               |true     |
|691       |1382      |2026-03-24 00:00:00|2026-03-24 00:00:00|NULL               |true     |
|250       |500       |2026-03-23 00:00:00|2026-03-23 00:00:

In [9]:
product_raw = spark.read.format("json").load(RAW_PATH + PRODUCTS)
product_raw.orderBy(sf.col("created_date").desc()).select("product_id", "created_at", "created_date").show(10, truncate=False)

+----------+----------+-------------------+
|product_id|created_at|created_date       |
+----------+----------+-------------------+
|1         |NULL      |2026-03-28 20:20:20|
|2         |NULL      |2026-03-28 20:20:20|
|3         |NULL      |2026-03-28 20:20:20|
|4         |NULL      |2026-03-28 20:20:20|
|5         |NULL      |2026-03-28 20:20:20|
|6         |NULL      |2026-03-28 20:20:20|
|7         |NULL      |2026-03-28 20:20:20|
|8         |NULL      |2026-03-28 20:20:20|
|9         |NULL      |2026-03-28 20:20:20|
|10        |NULL      |2026-03-28 20:20:20|
+----------+----------+-------------------+
only showing top 10 rows



In [17]:
spark.stop()